# Vue2 + Vue3 框架 · 第 03 天

## 生命周期、工程化开发与组件入门

> 本篇依据第 03 天 MD 笔记、PPT 和随堂代码整理。当天示例以 Vue 2 的 Options API 和 Vue CLI 为主，重点是理解生命周期、组件化和工程化的基本工作方式。

第 02 天我们学习了“数据变化后如何计算和响应”。第 03 天继续解决两个问题：

- **代码什么时候执行最合适？** 用生命周期钩子安排请求、DOM 操作和清理工作。
- **页面越来越大怎么办？** 用脚手架和组件把项目拆成可维护、可复用的模块。

## 一、学习目标与知识地图

| 模块 | 需要掌握的内容 | 典型场景 |
| --- | --- | --- |
| 生命周期 | 创建、挂载、更新、销毁四个阶段；8 个常用钩子 | 初始化请求、获取焦点、图表初始化、清理定时器 |
| 小黑记账清单 | 列表、添加、删除、计算总额、饼图 | 综合练习生命周期、计算属性、Axios、ECharts |
| 工程化 | 构建工具、Vue CLI、目录结构和运行流程 | 从 HTML 文件过渡到标准 Vue 项目 |
| 组件化 | 根组件、普通组件、单文件组件 | 拆分页面、复用模块、局部/全局注册 |
| 小兔仙首页 | 按页面模块拆分组件、复用通用按钮 | 练习组件边界和项目结构 |

### 本天最重要的判断

- 需要数据但不依赖 DOM：通常放在 `created`。
- 需要访问已经渲染出的 DOM：放在 `mounted`，或者数据更新后使用 `this.$nextTick()`。
- 需要销毁定时器、事件监听和图表实例：放在 `beforeDestroy`/`destroyed`（Vue 2）或 `beforeUnmount`/`unmounted`（Vue 3）。

## 二、Vue 生命周期

生命周期是一个 Vue 实例从创建到销毁的完整过程，可以分为四个阶段：

1. **创建阶段**：初始化实例、响应式数据、事件和生命周期钩子。
2. **挂载阶段**：把模板编译并渲染到真实 DOM。
3. **更新阶段**：数据改变，重新生成并更新 DOM。
4. **销毁阶段**：解除事件、销毁实例，释放资源。

![Vue 生命周期示意图](assets/day03/1682065937815.png)

### Vue 2 常用钩子顺序

```text
beforeCreate → created
              ↓
beforeMount → mounted
              ↓
beforeUpdate → updated（数据更新时循环）
              ↓
beforeDestroy → destroyed
```

Vue 3 的 Composition API 使用 `onBeforeMount`、`onMounted`、`onUpdated`、`onBeforeUnmount`、`onUnmounted` 等函数表达相同阶段；生命周期思想没有改变。

## 三、生命周期钩子详解

### 3.1 创建阶段

- `beforeCreate`：实例刚建立，`data`、`methods` 等还没有完成代理，通常很少使用。
- `created`：响应式数据和方法已经准备好，但 DOM 尚未挂载。适合发起初始化请求、准备数据、启动非 DOM 逻辑。

### 3.2 挂载阶段

- `beforeMount`：模板已经编译，但还没有插入真实 DOM。
- `mounted`：真实 DOM 已经生成并插入页面。适合获取输入框焦点、初始化 ECharts、读取 DOM 尺寸。

### 3.3 更新阶段

- `beforeUpdate`：数据变化，DOM 更新前触发。
- `updated`：DOM 更新后触发。不要在这里无条件修改响应式数据，否则可能造成更新循环。

### 3.4 销毁阶段

- `beforeDestroy`：实例销毁前，仍然可以访问数据和 DOM。适合清理定时器、取消监听、销毁第三方实例。
- `destroyed`：实例已经销毁，响应式连接和监听已经解除。

### 3.5 生命周期演示

```html
<div id="app">
  <h3>{{ title }}</h3>
  <button @click="count--">-</button>
  <span>{{ count }}</span>
  <button @click="count++">+</button>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: { title: '计数器', count: 100 },
  beforeCreate () { console.log('beforeCreate：data 还未代理') },
  created () { console.log('created：可以访问 this.count =', this.count) },
  beforeMount () { console.log('beforeMount：即将挂载 DOM') },
  mounted () { console.log('mounted：DOM 已经挂载') },
  beforeUpdate () { console.log('beforeUpdate：数据变了，DOM 还未更新') },
  updated () { console.log('updated：DOM 已更新') },
  beforeDestroy () { console.log('beforeDestroy：准备清理资源') },
  destroyed () { console.log('destroyed：实例已销毁') }
})
</script>
```

### 3.6 `created` 与 `mounted` 的选择

```js
created () {
  // 不需要页面元素，越早请求越快
  this.getList()
},
mounted () {
  // 需要真实 DOM，放在这里
  this.$refs.searchInput.focus()
}
```

如果数据更新后要读取新 DOM，使用：

```js
this.message = '新内容'
this.$nextTick(() => {
  console.log('DOM 已根据 message 更新')
})
```

## 四、生命周期案例

### 4.1 `created` 中发送初始化请求

页面需要一打开就显示新闻列表时，应该在 `created` 中请求数据：

```html
<div id="app">
  <ul>
    <li v-for="item in list" :key="item.id">
      <h3>{{ item.title }}</h3>
      <p>{{ item.source }} · {{ item.time }}</p>
      <img :src="item.img" alt="新闻图片">
    </li>
  </ul>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script src="https://cdn.jsdelivr.net/npm/axios/dist/axios.min.js"></script>
<script>
new Vue({
  el: '#app',
  data: { list: [] },
  created () {
    axios.get('http://hmajax.itheima.net/api/news')
      .then(res => { this.list = res.data.data })
      .catch(err => console.error('新闻加载失败', err))
  }
})
</script>
```

### 4.2 `mounted` 中获取焦点

```html
<div id="app">
  <input ref="searchInput" v-model="words" placeholder="输入搜索内容">
  <button @click="search">搜索</button>
</div>

<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script>
new Vue({
  el: '#app',
  data: { words: '' },
  mounted () {
    this.$refs.searchInput.focus()
  },
  methods: {
    search () { console.log('搜索：', this.words) }
  }
})
</script>
```

`ref` 是访问 DOM 或子组件实例的入口。它应该用于确实需要直接操作 DOM 的场景，普通显示和交互优先使用响应式数据。

## 五、综合案例：小黑记账清单

### 5.1 需求拆分

1. **基本渲染**：`created` 请求账单数据，`v-for` 渲染列表。
2. **添加**：`v-model.trim.number` 收集表单，校验后发送 `POST` 请求。
3. **删除**：拿到当前行 `id`，发送 `DELETE` 请求后重新加载。
4. **统计**：用 `computed` 计算消费总额。
5. **饼图**：在 `mounted` 中初始化 ECharts，列表改变后重新设置图表数据。

### 5.2 数据和接口

课程示例接口：

```text
GET    https://applet-base-api-t.itheima.net/bill
POST   https://applet-base-api-t.itheima.net/bill
DELETE https://applet-base-api-t.itheima.net/bill/:id
```

接口是否可用取决于练习环境；如果接口失效，可以把 `list` 换成静态数组，先完成 Vue 的渲染和交互。

### 5.3 记账清单核心实现

```html
<div id="app">
  <form @submit.prevent="addBill">
    <input v-model.trim="name" placeholder="消费名称">
    <input v-model.number="price" type="number" min="0" step="0.01" placeholder="消费价格">
    <button>添加账单</button>
  </form>

  <table>
    <thead><tr><th>编号</th><th>消费名称</th><th>消费价格</th><th>操作</th></tr></thead>
    <tbody>
      <tr v-for="item in list" :key="item.id">
        <td>{{ item.id }}</td>
        <td>{{ item.name }}</td>
        <td>{{ Number(item.price).toFixed(2) }}</td>
        <td><a href="#" @click.prevent="removeBill(item.id)">删除</a></td>
      </tr>
    </tbody>
    <tfoot><tr><td colspan="4">消费总计：{{ totalPrice.toFixed(2) }}</td></tr></tfoot>
  </table>

  <div id="main" style="width:600px;height:400px"></div>
</div>

<script src="https://cdn.jsdelivr.net/npm/echarts@5.4.0/dist/echarts.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
<script src="https://cdn.jsdelivr.net/npm/axios/dist/axios.min.js"></script>
<script>
new Vue({
  el: '#app',
  data: {
    list: [], name: '', price: null, chart: null
  },
  computed: {
    totalPrice () {
      return this.list.reduce((sum, item) => sum + Number(item.price), 0)
    }
  },
  created () {
    this.loadList()
  },
  mounted () {
    this.chart = echarts.init(document.querySelector('#main'))
    this.updateChart()
  },
  methods: {
    async loadList () {
      try {
        const res = await axios.get('https://applet-base-api-t.itheima.net/bill')
        this.list = res.data.data
        this.updateChart()
      } catch (error) {
        console.error('账单加载失败', error)
      }
    },
    async addBill () {
      if (!this.name) return alert('请输入消费名称')
      if (!Number.isFinite(this.price) || this.price <= 0) return alert('请输入正确的消费价格')
      await axios.post('https://applet-base-api-t.itheima.net/bill', {
        name: this.name, price: this.price
      })
      this.name = ''
      this.price = null
      await this.loadList()
    },
    async removeBill (id) {
      await axios.delete(`https://applet-base-api-t.itheima.net/bill/${id}`)
      await this.loadList()
    },
    updateChart () {
      if (!this.chart) return
      this.chart.setOption({
        title: { text: '消费账单列表' },
        tooltip: { trigger: 'item' },
        series: [{
          name: '消费账单', type: 'pie', radius: '50%',
          data: this.list.map(item => ({ name: item.name, value: Number(item.price) }))
        }]
      })
    }
  },
  beforeDestroy () {
    if (this.chart) this.chart.dispose()
  }
})
</script>
```

关键点：`created` 可以请求数据，`mounted` 才能安全地初始化 `#main`。数据回来后先更新 `list`，再调用 `setOption` 更新图表；真实项目中还应处理加载中、空数据和请求失败状态。

## 六、工程化开发与脚手架

### 6.1 两种开发方式

#### 核心包传统开发

直接在 HTML 中引入 Vue、Axios 等核心包：

```html
<script src="https://cdn.jsdelivr.net/npm/vue@2/dist/vue.js"></script>
```

适合教学、局部功能改造和小型页面。文件之间的依赖、模块化和构建优化需要自己维护。

#### 工程化开发

在 Webpack 等构建工具提供的环境中开发 Vue。源代码可以使用 ES Module、单文件组件、Less、TypeScript 等，再由构建工具转换成浏览器可运行的资源。

工程化的价值：

- 模块化组织代码，依赖关系清晰。
- 统一处理编译、压缩、打包和开发服务器。
- 支持更现代的 JS、CSS 和 TypeScript 语法。
- 通过脚手架生成统一、可维护的项目结构。

### 6.2 Vue CLI

课程使用 Vue CLI，它是 Vue 官方提供的全局命令工具，可快速生成集成了构建配置的标准项目。

```bash
# 安装一次即可
npm install --global @vue/cli

# 查看版本
vue --version

# 创建项目，项目名不要使用中文
vue create vue-demo

# 进入目录并启动开发服务器
cd vue-demo
npm run serve
```

实际启动命令以项目 `package.json` 的 `scripts` 为准。新项目也可能使用 Vite，命令通常是 `npm run dev`；学习重点是理解“脚手架负责构建，组件负责组织页面”。

## 七、脚手架项目目录与运行流程

### 7.1 三个最先掌握的文件

- `public/index.html`：页面模板，通常只有一个 `#app` 挂载点。
- `src/main.js`：入口文件，创建 Vue 实例并挂载根组件。
- `src/App.vue`：根组件，包裹整个应用的普通组件。

典型 Vue 2 入口：

```js
import Vue from 'vue'
import App from './App.vue'

Vue.config.productionTip = false

new Vue({
  render: h => h(App)
}).$mount('#app')
```

### 7.2 运行流程

```text
npm run serve
    ↓
构建工具读取 main.js
    ↓
main.js 导入 App.vue
    ↓
App.vue 组合普通组件
    ↓
模板编译为 render 函数
    ↓
挂载到 public/index.html 的 #app
    ↓
浏览器显示页面
```

![工程化目录与运行流程](assets/day03/1682092148521.png)

## 八、组件化开发

组件化是把一个页面拆成多个独立模块，每个组件拥有自己的结构、样式和行为。组件之间通过明确的输入和输出协作。

### 8.1 组件的分类

- **根组件**：通常是 `App.vue`，位于应用最上层。
- **普通组件**：如页头、导航、商品卡片、页脚、通用按钮等。

### 8.2 组件化的好处

- 结构更清晰，修改一个模块时影响范围更小。
- 同一组件可以在多个页面复用。
- 每个组件可以独立维护自己的样式和逻辑。
- 多人协作时可以按组件拆分任务。

### 8.3 单文件组件 `.vue`

```vue
<template>
  <div class="hm-header">我是页头</div>
</template>

<script>
export default {
  name: 'HmHeader'
}
</script>

<style scoped>
.hm-header {
  height: 100px;
  line-height: 100px;
  text-align: center;
  color: #fff;
  background: #8064a2;
}
</style>
```

一个单文件组件通常包含三部分：

- `template`：结构，一个组件只能有一个根元素（课程使用的 Vue 2 版本要求）。
- `script`：数据、方法、计算属性、生命周期等逻辑。
- `style`：样式；`scoped` 可以把样式限制在当前组件。

需要 Less 时，使用 `<style lang="less">`，并安装对应的 `less`、`less-loader` 依赖。

## 九、局部注册组件

### 9.1 特点

局部注册的组件只能在注册它的组件模板中使用，作用域清晰，适合页面专用模块。

### 9.2 步骤

1. 创建 `.vue` 文件。
2. 在使用方通过 `import` 导入。
3. 在 `components` 配置项注册。
4. 在模板中用组件标签使用。

```vue
<!-- src/components/HmHeader.vue -->
<template>
  <header class="hm-header">我是页头</header>
</template>

<script>
export default { name: 'HmHeader' }
</script>
```

```vue
<!-- App.vue -->
<template>
  <div>
    <HmHeader />
    <main>我是主体</main>
  </div>
</template>

<script>
import HmHeader from './components/HmHeader.vue'

export default {
  components: {
    HmHeader
  }
}
</script>
```

组件命名推荐使用大驼峰，例如 `HmHeader`、`GoodsItem`。在模板中可以写 `<HmHeader />` 或 `<hm-header></hm-header>`，项目中保持一种风格即可。

## 十、全局注册组件

### 10.1 特点

全局注册后，项目中的任意组件都可以直接使用。通用按钮、图标、弹窗等多个页面都会用到的组件适合全局注册。

### 10.2 在 `main.js` 注册

```js
import Vue from 'vue'
import App from './App.vue'
import HmButton from './components/HmButton.vue'

Vue.component('HmButton', HmButton)

new Vue({ render: h => h(App) }).$mount('#app')
```

```vue
<!-- 任意组件中都可以使用 -->
<template>
  <div>
    <HmButton>保存</HmButton>
  </div>
</template>
```

全局注册很方便，但过多使用会让依赖关系不明显。页面专用组件优先局部注册，真正通用的基础组件再考虑全局注册。

## 十一、综合案例：小兔仙首页拆分

课程的拆分思路是先看页面模块，再决定组件边界：

1. 分析设计图，把页头、导航、Banner、商品分类、推荐区域、页脚等模块拆开。
2. 先写出每个组件的静态结构和样式，让页面骨架跑起来。
3. 页面专用模块局部注册，通用按钮或通用卡片可以全局注册。
4. 后续再通过 `props`、事件和路由把静态页面变成可交互页面。

推荐的目录结构：

```text
src/
├─ App.vue
├─ main.js
├─ components/
│  ├─ HmHeader.vue
│  ├─ HmNav.vue
│  ├─ HmBanner.vue
│  ├─ GoodsItem.vue
│  └─ HmFooter.vue
├─ views/
│  └─ HomeView.vue
└─ styles/
   ├─ base.css
   └─ common.css
```

拆分组件时要让每个组件有清楚的职责：页头只负责页头结构，商品项只负责一件商品的展示，页面组件负责组合它们。不要为了“看起来组件很多”而把一小段没有复用价值的标签强行拆成组件。

## 十二、常见错误与排查

### 12.1 在 `created` 中操作 DOM

`created` 时 DOM 还没有挂载，`document.querySelector` 可能取不到元素。需要 DOM 时改用 `mounted` 或 `$nextTick`。

### 12.2 ECharts 初始化时机不对

图表容器必须有真实宽高，并且已经挂载：

```js
mounted () {
  this.chart = echarts.init(this.$refs.chart)
}
```

组件销毁前要调用 `dispose()`，避免重复初始化和内存泄漏。

### 12.3 请求重复发送

不要在 `updated` 中无条件请求接口。数据更新会触发 `updated`，请求回来再次改数据又会触发更新，容易形成循环。初始化请求放 `created`，特定字段变化请求放 `watch`。

### 12.4 局部组件找不到

检查三件事：路径是否正确、是否完成 `import`、是否写入 `components` 配置项。注册名和模板标签要对应。

### 12.5 组件样式相互污染

组件样式默认可能影响其他组件；页面专用样式建议使用 `<style scoped>`，全局基础样式单独放在公共 CSS 文件中。

## 十三、当天练习

### 练习 1：生命周期日志

创建一个计数器，分别在 8 个 Vue 2 生命周期钩子中打印日志，点击按钮观察更新阶段的执行顺序。

### 练习 2：新闻列表

用 `created` 请求新闻接口，增加加载中、空列表和错误提示。点击新闻时打印当前新闻的 id。

### 练习 3：记账清单增强

给账单增加“编辑”功能：点击编辑后把当前数据放回表单，提交时使用 `PUT` 更新，再重新加载列表和饼图。

### 练习 4：组件拆分

创建 `HmHeader`、`HmMain`、`HmFooter` 三个局部组件，在 `App.vue` 中组合成页面；再创建 `HmButton` 并改为全局注册。

### 练习 5：销毁清理

在组件中创建一个 `setInterval`，在 `beforeDestroy` 中清除它。打开和关闭组件多次，确认定时器不会叠加。

## 十四、当天速记

- Vue 生命周期是实例从创建、挂载、更新到销毁的过程。
- 初始化请求通常放 `created`，访问真实 DOM 或初始化图表放 `mounted`。
- 数据更新后的 DOM 操作使用 `this.$nextTick()`。
- 定时器、事件监听和第三方实例要在销毁前清理。
- 工程化用脚手架统一处理构建和项目配置；入口通常是 `main.js`，根组件是 `App.vue`。
- 单文件组件由 `template`、`script`、`style` 三部分组成。
- 局部注册作用域小、依赖清楚；全局注册适合真正通用的基础组件。
- 页面开发先拆模块，再写结构样式，最后补充数据和交互。